# Hyperparameter Tuning

For the past 15 lessons, we have allowed our algorithms to use default settings. We let `scikit-learn` decide that our Random Forest should have `n_estimators=100` or that our SVM should use a `C=1.0`. 

In the enterprise world, relying on default settings is equivalent to buying a high-performance sports car and never shifting it out of second gear. To squeeze the absolute maximum mathematical performance out of an algorithm, we must tune its **Hyperparameters**.

Before we begin, we must establish a strict mathematical distinction that is often confused by junior developers:
* **Parameters ($\theta$, $w$, $b$)**: Variables that the algorithm learns *automatically* from the data during the training phase (e.g., the slope of a line, the splits in a decision tree).
* **Hyperparameters ($\alpha$, $C$, $K$, $\nu$)**: Variables that govern the architecture and the learning process itself. The algorithm *cannot* learn these from the data. You, the human engineer, must set them before training begins.

Because we cannot use Gradient Descent to solve for hyperparameters, we must use **Meta-Optimization Search Strategies**. In this lesson, we will explore Grid Search, Random Search, and the highly advanced Bayesian Optimization.

Let's set up our Python environment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Meta-Optimization Environment Ready.")

✅ Meta-Optimization Environment Ready.


# 1. Grid Search (Exhaustive Brute Force)

The most intuitive way to find the best hyperparameters is to literally test every single possible combination. This is called **Grid Search**.

You define a discrete set of values for each hyperparameter. The algorithm takes the Cartesian product of these sets, trains a separate model for every single combination using K-Fold Cross-Validation, and returns the one with the highest accuracy.

### The Mathematics of the Grid
Let $H_1$ be the number of trees (e.g., $\{10, 50, 100\}$). Let $H_2$ be the maximum depth (e.g., $\{3, 5, 10\}$).
The total search space $S$ is:
$$|S| = |H_1| \times |H_2| = 3 \times 3 = 9 \text{ models}$$

**The Fatal Flaw (The Curse of Dimensionality):**
Grid Search suffers from exponential explosion. If you have 5 hyperparameters and want to test 10 values for each, the search space becomes $10^5 = 100,000$ models. If each model takes 1 minute to train, a Grid Search will take **69 days** to finish. 

# 2. Random Search (Stochastic Optimization)

In 2012, researchers Bergstra and Bengio published a landmark paper proving that **Random Search** is mathematically superior to Grid Search in high-dimensional spaces. 

Instead of exhaustively testing every combination, you define a continuous statistical distribution (e.g., a Uniform Distribution between $0.01$ and $1.0$) and the algorithm randomly samples $N$ combinations.

### Why is Random Search Better? (Low Effective Dimensionality)
In almost every Machine Learning model, not all hyperparameters are equally important. 
Imagine a 2D search space where Hyperparameter A is critically important, and Hyperparameter B is completely useless. 

* **Grid Search (3x3 = 9 trials)**: Even though you ran 9 models, you only tested **3 unique values** of the important Hyperparameter A. You wasted 6 runs testing combinations of the useless Hyperparameter B.
* **Random Search (9 trials)**: Because the coordinates are continuous and random, you test **9 unique values** of the important Hyperparameter A. 

Random Search explores the critical axes of your cost function exponentially faster than Grid Search!

# 3. Implementing the Search in Code

Let's simulate a complex classification problem and use `RandomizedSearchCV` to tune a Random Forest, finding the absolute perfect balance of bias and variance.

In [2]:
# 1. Simulate Corporate Dataset
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the Base Algorithm
rf = RandomForestClassifier(random_state=42)

# 3. Define the Statistical Distributions for the Hyperparameters
param_distributions = {
    'n_estimators': randint(50, 500),              # Discrete Uniform: 50 to 500 trees
    'max_depth': randint(3, 20),                   # Discrete Uniform: Depth 3 to 20
    'min_samples_split': randint(2, 20),           # Prevent overfitting at nodes
    'max_features': uniform(0.1, 0.9)              # Continuous Uniform: 10% to 100% of features
}

# 4. Initialize Randomized Search
# n_iter=50 means we will randomly sample 50 unique combinations
# cv=3 means each combination is cross-validated 3 times (150 total training runs!)
random_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_distributions, 
    n_iter=50, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1, # Use all available CPU cores to parallelize
    random_state=42
)

print("🚀 Initiating Meta-Optimization (Training 150 models in parallel)...")
random_search.fit(X_train, y_train)

# 5. Extract the Results
print(f"✅ Search Complete!")
print(f"🚨 Best Cross-Validation Accuracy: {random_search.best_score_ * 100:.2f}%")
print(f"🚨 Mathematically Optimal Hyperparameters:\n {random_search.best_params_}")

# Evaluate the best model on the locked Test Set
best_model = random_search.best_estimator_
test_acc = best_model.score(X_test, y_test)
print(f"🚨 Final Exam (Test Set) Accuracy: {test_acc * 100:.2f}%")

🚀 Initiating Meta-Optimization (Training 150 models in parallel)...
✅ Search Complete!
🚨 Best Cross-Validation Accuracy: 90.25%
🚨 Mathematically Optimal Hyperparameters:
 {'max_depth': 14, 'max_features': np.float64(0.3266040662428278), 'min_samples_split': 3, 'n_estimators': 436}
🚨 Final Exam (Test Set) Accuracy: 90.00%


# 4. Expert Level: Bayesian Optimization

Both Grid Search and Random Search share a massive fundamental flaw: **They are completely blind.**
If Random Search tests a Learning Rate of $0.5$ and it results in an awful accuracy of $20\%$, the algorithm learns nothing. It completely erases that failure from its memory and blindly picks the next random number.

**Bayesian Optimization (SMBO)** fixes this. It uses AI to optimize the AI. 

### The Mathematics of Intelligent Search
1.  **The Surrogate Model**: We use a probabilistic algorithm (usually a **Gaussian Process**) to build a mathematical map of the Hyperparameter Cost Function. It maps out where it *thinks* the highest accuracy is, along with an uncertainty zone (variance).
2.  **The Acquisition Function**: We use a calculus formula, usually **Expected Improvement (EI)**, to decide which hyperparameter to test next. It balances:
    * **Exploitation**: Testing hyperparameters near a known "good" area.
    * **Exploration**: Testing hyperparameters in a completely unknown area of the map just in case a massive spike in accuracy is hiding there.

By learning from its past mistakes, Bayesian Optimization can find the absolute optimal hyperparameters in 30 intelligent attempts, whereas Random Search might require 1,000 blind attempts. (Libraries like `Optuna` or `Hyperopt` are the enterprise standard for this).

## Real-World Use Case or Analogy:
Think of Hyperparameter Tuning like **Cracking a Safe**:

* **Grid Search**: You start at `00-00-00` and methodically turn the dial to `00-00-01`, `00-00-02`, all the way to `99-99-99`. It is mathematically guaranteed to find the right code, but you will die of old age before it opens.
* **Random Search**: You spin the dials completely at random. Statistically, you will likely hit a combination much closer to the true code faster than starting at zero and moving sequentially.
* **Bayesian Optimization**: You put a stethoscope to the safe. You try a combination, and you *listen* for the click of the tumbler. You use that sound (past information) to intelligently guide which direction you should turn the dial next. You crack the safe in 5 minutes.

---